In [1]:
from langchain_ollama import ChatOllama
from langchain_core.prompts import ChatPromptTemplate
from sqlalchemy import create_engine
import pandas as pd

In [3]:
username = "root"
password = "jessepinkman"
host = "localhost"
database = "company_db"

In [4]:
engine = create_engine(f"mysql+pymysql://{username}:{password}@{host}/{database}")

In [5]:
llm = ChatOllama(model="llama3")

In [6]:
schema = """
Table: employees
Columns: employee_id, employee_name, age, gender, department_id, salary, joining_date, city

Table: departments
Columns: department_id, department_name

Table: customers
Columns: customer_id, customer_name, gender, age, city, phone_number, email, registration_date

Table: products
Columns: product_id, product_name, category, brand, price, stock_quantity

Table: orders
Columns: order_id, customer_id, product_id, employee_id, quantity, order_amount, order_date, payment_method, order_status

Table: sales
Columns: sale_id, employee_id, product_name, sale_amount, sale_date, region
"""

In [7]:
prompt = ChatPromptTemplate.from_template(
    """
    You are an expert MySQL query generator.

    Database Schema:
    {schema}

    User Question:
    {question}

    Rules:
    1. Return only SQL query
    2. Do not explain anything
    3. Use only given tables and columns
    """
)

In [13]:
user_question = "give me total count of employees in every department and sort by descending order"

In [14]:
formatted_prompt = prompt.format(
    schema=schema,
    question=user_question
)

response = llm.invoke(formatted_prompt)

In [15]:
generated_sql = response.content.strip()

print(generated_sql)

SELECT d.department_name, COUNT(e.employee_id) AS total_employees 
FROM departments d 
JOIN employees e ON d.department_id = e.department_id 
GROUP BY d.department_name 
ORDER BY total_employees DESC;


In [16]:
data = pd.read_sql(generated_sql, engine)

In [17]:
data

,department_name,total_employees
0,Data,5
1,IT,4
2,Sales,4
3,HR,4
4,Marketing,4
5,Finance,3
6,Legal,3
7,Operations,3


In [18]:
#adding the concept of chart generation

In [19]:
chart_keywords = [
    "chart",
    "graph",
    "plot",
    "trend",
    "compare",
    "distribution",
    "by month",
    "by region",
    "by department"
]

In [21]:
chart_prompt = f"""
User Question: {user_question}

Should this result be shown as a chart?

Only answer YES or NO.
"""

In [22]:
chart_response = llm.invoke(chart_prompt)

needs_chart = chart_response.content.strip().upper()

print(needs_chart)

NO


In [24]:
if needs_chart == "YES":
    print("Chart is needed")
else:
    print("No Need of Chart")

No Need of Chart


In [25]:
import matplotlib.pyplot as plt

In [27]:
if needs_chart == "YES":
    print(data.columns)

In [28]:
if needs_chart == "YES":
    
    x_column = data.columns[0]
    y_column = data.columns[1]
    
    plt.figure(figsize=(8,5))
    plt.bar(data[x_column], data[y_column])
    
    plt.title(user_question)
    plt.xlabel(x_column)
    plt.ylabel(y_column)
    
    plt.xticks(rotation=45)
    plt.show()